<a href="https://colab.research.google.com/github/yumna-09/FlyRank-ML-Internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yumna-09/FlyRank-ML-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question:** When a content site has far more pages than a review team can check by
hand, which pages should a human look at first — and does grouping pages by *observed behavior*
(impressions, CTR, ranking position, engagement, age) sharpen that priority list more than a
single hand-written rule can?

**Lane:** Structured Content Archetype Clustering (unsupervised — there is no predefined label;
the groups themselves are the output).

**Decision it supports:** which pages a content strategist reviews first each week — refresh,
boost, prune, or leave alone.

**Who acts on it:** a content strategist or SEO reviewer with limited hours per week, working
through a catalog of hundreds of thousands of pages.

**Cost of a wrong call:** a page wrongly read as "healthy" gets ignored and keeps losing traffic
silently; a page wrongly flagged gets editorial time it didn't need, and a genuinely valuable
page could be pruned by mistake. Both failure modes cost real time or real traffic — which is
why every recommendation in this paper is decision-support for a human, never an automated action.

In [ ]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "FlyRank-ML-Internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "https://github.com/yumna-09/FlyRank-ML-Internship.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df_preview = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Why grouping beats one rule — a quick look:")
print("\nContent type volume:")
print(df_preview["content_type"].value_counts().head(3))
print(f"\nimpressions_90d spread: min={df_preview['impressions_90d'].min()}, "
      f"median={df_preview['impressions_90d'].median():.0f}, "
      f"75th pct={df_preview['impressions_90d'].quantile(0.75):.0f}, "
      f"max={df_preview['impressions_90d'].max()}")
print("\ntrend_direction distribution:")
print(df_preview["trend_direction"].value_counts())

Why grouping beats one rule — a quick look:

Content type volume:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

impressions_90d spread: min=1, median=731, 75th pct=3615, max=517715

trend_direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** `FlyRank/internship-warehouse` on Hugging Face (gated, instant approval), build id
`flyrank_pseudonymized_warehouse_release_v20260703`, exported `2026-07-03` from FlyRank's real
production search-and-analytics warehouse (Google Search Console + Google Analytics, synced
daily), pseudonymized before release — no client names, domains, URLs, page titles, or raw
search queries anywhere in it.

**Scale of the release:** the daily fact table (`fact_content_daily_performance`) alone holds
**78,835,655 rows** — daily × client × content — across 104 clients and 519,606 content items.
That is the production-scale evidence this whole track is trained on.

**What I actually pulled from it:**
- **Data-contract exercise (Week 4):** a March 2026 slice queried directly from the warehouse
  with DuckDB — 9,841,378 rows, 331,437 distinct content items, 55 clients, `report_date`
  2026-03-01 to 2026-03-31. Used to practice grain verification and a real leakage trap (see
  Methodology), not the main modeling pipeline.
- **Main clustering + playbook pipeline (Weeks 5-7, this paper's results):** the bundled
  anonymized starter sample, `data/raw/content_refresh_anonymized.csv` — 30,000 rows, one row
  per pseudonymized content item, pre-aggregated 90-day metrics. Smaller than the full release
  by design (Hugging Face rate limits, and a stable slice to iterate on).

**Excluded, and why:**
- `ga4_*` columns where `ga4_data_available` is not `TRUE` (95.8% of the March slice) — a
  zero there means "not measured yet," not "zero engagement." Treating it as a real zero would
  teach a model a false pattern.
- FlyRank's own product decision flags (`health_score`, `priority_score`, `action_type`) — not
  shipped in this release, and would be circular as inputs: a model trained on them would just
  reproduce an existing rule instead of finding anything new. They're a comparison target, never
  a feature.
- `trend_direction` / `trend_pct` — the label source for "is this page declining?" — excluded
  from every feature list; used only afterward, descriptively, never for clustering itself.
- `content_id` / `client_id` — pseudonymous hashes, used only for joins and the client-grouped
  train/test split, never fed to any model as a value.

In [ ]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "FlyRank-ML-Internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "https://github.com/yumna-09/FlyRank-ML-Internship.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Starter sample shape:", df.shape)
print("Distinct clients in starter sample:", df["client_id"].nunique())
print("Declining-label rate:", round((df["trend_direction"] == "down").mean(), 4))

Starter sample shape: (30000, 44)
Distinct clients in starter sample: 32
Declining-label rate: 0.5421


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumption:** pages don't fail on one dimension at a time — a page can have high impressions
and a bad CTR, or low impressions and strong engagement. A single if-this-then-that rule can
only test one or two conditions before it needs dozens of nested branches to cover real
combinations. Clustering looks at five signals together and lets the groups emerge.

**Features (5, all trailing 90-day observed metrics, no leakage):** `impressions_90d`, `ctr`,
`avg_position`, `engagement_rate`, `content_age_days`. Scaled with `StandardScaler` before
clustering so `impressions_90d` (scale: thousands) doesn't dominate `ctr` (scale: single digits).

**Label / proxy:** none used for clustering — it's unsupervised. `is_declining_label`
(`trend_direction == "down"`) exists in the data and is used *afterward*, descriptively, to
check whether clusters differ in decline rate — never as a clustering input.

**Baseline (built first, Week 4):** a transparent rule, no fitted weights. Two real signals
were tested against the data before either went into the score:
- *Staleness → decline?* **Mixed** — decline rate rises from 51.1% (freshest bucket) to 61.1%
  (91-180 days stale) but then *drops* to 47.1% in the oldest bucket (181+ days, n=174) — the
  opposite of what the "stale pages decay more" assumption predicts. Dropped from the rule.
- *Position → CTR?* **Confirmed** — median CTR falls in lock-step with position tier, every
  bucket well-sampled (n=389 to n=7,064): `top_3` 0.20% → `page_1` 0.24% → `striking` 0.17% →
  `page_3_5` 0.09% → `deep` 0.00%. Kept as the rule's core signal.

The baseline score = `visibility × ctr_gap`, both percentile-ranked, restricted to eligible
pages (`impressions_90d ≥ 500`, ranked, has a position tier) — three action buckets from two
fixed thresholds, no model fitting.

**Model (Week 5):** KMeans, k chosen by silhouette search over k=2..7 (k=7 won). Chosen over a
supervised model because this lane has no target to predict, and KMeans is simple, fast, and
its output — cluster membership — is easy to explain to a non-technical reviewer.

**Validation design (Week 6, honest re-check):** the Week-5 number was leaky — centroids were
fit AND scored on the same 30,000 rows. Honest fix: split the **55 client IDs** (not rows)
80/20, fit KMeans centroids on the 80% train-client rows only, then score strictly on the 20%
held-out clients' rows the model never touched. This tests whether the archetypes generalize to
a brand-new client's pages, which is the real-world use case.

**Leakage checks run:**
1. *k-selection leakage* — was `k=7` chosen using the full dataset, then "validated" on that
   same full dataset? Re-ran the silhouette search on train-clients-only data: `k=7` won there
   too. No leakage found — the winning k is stable with or without the held-out clients.
2. *Feature leakage* — confirmed `client_id` / `content_id` are not in the feature list, and
   `trend_direction` / `trend_pct` (the label source) never enter the clustering features.
3. *Label leakage* (data-contract exercise, Week 4) — a toy label built purely to test this:
   adding a feature computed from the same window as the label pushed accuracy from a believable
   0.591 to a suspicious 1.000. That leaking feature was deleted; the real pipeline in this paper
   never uses second-half-of-window features to explain first-half outcomes.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# --- Baseline (Week 4): CTR-gap rule, no fitted weights ---
eligible = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["position_tier"] != "no_data")
expected_ctr_by_tier = df[eligible].groupby("position_tier")["ctr"].median()
df["expected_ctr"] = df["position_tier"].map(expected_ctr_by_tier)
df["ctr_gap"] = (df["expected_ctr"] - df["ctr"]).clip(lower=0)

def pct_rank(s):
    return s.rank(method="average", pct=True)

df["ctr_gap_norm"] = pct_rank(df["ctr_gap"].where(eligible, 0.0))
df["visibility_score"] = pct_rank(np.log1p(df["impressions_90d"]))
df["baseline_action_score"] = np.where(eligible, df["visibility_score"] * df["ctr_gap_norm"], 0.0).round(4)

def action_label(s):
    if s >= 0.6: return "prioritize_ctr_review"
    if s > 0: return "review_ctr"
    return "monitor"

df["action"] = df["baseline_action_score"].apply(action_label)

# --- Model (Week 5/6): KMeans, k chosen by silhouette search, honest client-grouped re-check ---
features = ["impressions_90d", "ctr", "avg_position", "engagement_rate", "content_age_days"]
X = df[features].dropna()
X_scaled = StandardScaler().fit_transform(X)

BEST_K = 7  # winner of the k=2..7 silhouette search in Week 5, confirmed stable in Week 6's leakage check
km_full = KMeans(n_clusters=BEST_K, random_state=42, n_init=10).fit(X_scaled)
sil_full_leaky = silhouette_score(X_scaled, km_full.labels_)

baseline_sil = silhouette_score(X_scaled, df.loc[X.index, "action"])

groups = df.loc[X.index, "client_id"]
rng = np.random.RandomState(42)
unique_clients = np.array(groups.unique())
rng.shuffle(unique_clients)
split_point = int(len(unique_clients) * 0.8)
train_clients = set(unique_clients[:split_point])
train_mask = groups.isin(train_clients).values
test_mask = ~train_mask

km_train = KMeans(n_clusters=BEST_K, random_state=42, n_init=10).fit(X_scaled[train_mask])
test_labels = km_train.predict(X_scaled[test_mask])
sil_honest = silhouette_score(X_scaled[test_mask], test_labels)

print("=== Section 3/4: Model vs Baseline, same feature space ===")
print(f"Baseline rule-buckets silhouette:        {baseline_sil:.4f}")
print(f"KMeans (all data, leaky):                {sil_full_leaky:.4f}")
print(f"KMeans (honest, held-out clients):       {sil_honest:.4f}")
print(f"Train clients: {len(train_clients)} | Held-out clients: {len(unique_clients) - len(train_clients)}")

=== Section 3/4: Model vs Baseline, same feature space ===
Baseline rule-buckets silhouette:        0.0013
KMeans (all data, leaky):                0.3763
KMeans (honest, held-out clients):       0.3046
Train clients: 25 | Held-out clients: 7


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Same feature space, same 30,000 rows — rule-based buckets vs. KMeans:

<table style="width:100%; border-collapse: collapse; text-align: left; font-family: sans-serif; font-size: 13px; color: #000000; background-color: #ffffff; table-layout: fixed; margin-top: 10px; margin-bottom: 15px;">
  <thead>
    <tr style="border-bottom: 2px solid #333333; background-color: #ffffff;">
      <th style="padding: 8px; width: 40%;">Method</th>
      <th style="padding: 8px; width: 18%; text-align: right;">Silhouette score</th>
      <th style="padding: 8px; width: 42%;">What it means</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #f4f4f4; border-bottom: 1px solid #e0e0e0;">
      <td style="padding: 8px;">Baseline: Week-4 rule buckets (<code>monitor</code> / <code>review_ctr</code> / <code>prioritize_ctr_review</code>)</td>
      <td style="padding: 8px; text-align: right;"><b>0.0013</b></td>
      <td style="padding: 8px;">essentially no natural separation — the buckets weren't built to group by <i>behavior</i>, they were built to flag one specific problem (CTR-vs-position)</td>
    </tr>
    <tr style="background-color: #ffffff; border-bottom: 1px solid #e0e0e0;">
      <td style="padding: 8px;">KMeans (k=7), fit and scored on all data (<b>leaky</b>)</td>
      <td style="padding: 8px; text-align: right;">0.3763</td>
      <td style="padding: 8px;">strong separation, but inflated — same rows shaped the centroids and the score</td>
    </tr>
    <tr style="background-color: #f4f4f4; border-bottom: 1px solid #e0e0e0;">
      <td style="padding: 8px;">KMeans (k=7), centroids fit on 80% of clients, scored on the other 20% (<b>honest</b>)</td>
      <td style="padding: 8px; text-align: right;"><b>0.3046</b></td>
      <td style="padding: 8px;">the trustworthy number — separation holds up on clients the model never saw</td>
    </tr>
  </tbody>
</table>

A silhouette score isn't a percent-correct metric, so there's no "base rate" to report the way a classifier would — the honest comparison here is method vs. method on the identical feature space, which is what the table shows: the rule buckets barely separate the data at all (0.0013), while KMeans — even penalized for held-out clients — still finds real structure (0.3046).

**Cluster archetypes (profiled on the full 30,000 rows):** Cluster 6 is a small group of very high-traffic top performers (avg. 112k impressions/90d). Clusters 0 and 1 are the largest groups — steady, older content with moderate impressions, differing mainly by age. Cluster 4 is low-CTR, poorly-ranked content (avg. position 46). Clusters 2 and 3 are small, unusual groups — one near-zero-impression/high-CTR (low-volume, highly targeted queries), one unusually high-engagement relative to its traffic. About 2.6% of all points have a negative silhouette (poorly matched to their cluster) — concentrated in a borderline-engagement group sitting between two clearer archetypes.

**Decline rate by cluster, on the 7 held-out clients only (descriptive, label not used in clustering):** ranges from 14.6% (Cluster 6, the top performers) to 61.9% (Cluster 2). This is an *observed association*, not proof the cluster causes decline, and 7 clients is a small sample — see Limitations.

In [ ]:
all_labels = km_train.predict(X_scaled)  # honest model, applied to every row for a full profile
df.loc[X.index, "cluster"] = all_labels

cluster_profile = df.loc[X.index].groupby("cluster")[features].mean().round(2)
cluster_profile["count"] = df.loc[X.index].groupby("cluster").size()
print("=== Cluster archetype profile (all 30,000 rows) ===")
print(cluster_profile)

test_idx = X.index[test_mask]
decline_test = (df.loc[test_idx, "trend_direction"] == "down").astype(int)
cluster_decline = pd.DataFrame({"cluster": test_labels, "decline": decline_test.values})
decline_rate = cluster_decline.groupby("cluster")["decline"].agg(["mean", "count"]).sort_values("mean")
decline_rate["decline_pct"] = (decline_rate["mean"] * 100).round(1)

print("\n=== Decline rate by cluster, held-out clients only (descriptive) ===")
print(decline_rate[["decline_pct", "count"]])

=== Cluster archetype profile (all 30,000 rows) ===
         impressions_90d    ctr  avg_position  engagement_rate  \
cluster                                                          
0.0              4019.57   0.40         11.55             1.52   
1.0            111339.51   0.34         10.66             3.12   
2.0              3937.60   0.31         11.58             1.16   
3.0              2623.43   0.10         46.89             0.90   
4.0               406.97   3.50         18.21            97.69   
5.0              1939.03   0.54         15.05            27.74   
6.0                 4.32  41.68          6.31             6.71   

         content_age_days  count  
cluster                           
0.0                385.39   9857  
1.0                263.24    421  
2.0                152.62  14495  
3.0                308.46   3962  
4.0                292.81    101  
5.0                265.33   1034  
6.0                285.35    130  

=== Decline rate by cluster, held-out

## 5. Limitations

*What this work cannot claim.*

- **Cross-sectional, not causal.** Every number here describes an association in one snapshot
  of data. No page was actually refreshed and re-measured, so nothing here can say that acting
  on a recommendation *will* improve a page's traffic — only that certain pages look worth
  reviewing first, given what's observed about similar pages.
- **Not "predicting Google."** Clustering groups pages that behave alike on 5 measured signals.
  It says nothing about the search-engine ranking algorithm itself, and archetype membership is
  behavioral, not topical — this is not semantic clustering (no page text was used).
- **Small held-out sample for the decline-rate finding.** The 14.6%-61.9% decline-rate range by
  cluster is measured on 7 held-out clients (7,611 pages) — real, but too small a client sample
  to say the pattern generalizes to FlyRank's full 104-client base.
- **Starter sample, not the full catalog.** The main pipeline (Weeks 5-7) ran on the 30,000-row
  anonymized starter sample, a slice of a 519,606-item catalog. Cluster shapes and thresholds
  here have not been re-validated against the full release.
- **Baseline wasn't tested under the same honest split as the model.** The 0.0013 vs 0.3763/0.3046
  comparison in Section 4 uses the same feature space and the same rows, but only KMeans was
  re-scored under the client-held-out split — the rule buckets were evaluated once, on all data.
- **Single snapshot, not a real time series.** `trend_direction` itself is a 30-vs-prior-30-day
  comparison inside one export — "decay" here means "declining in this one window," observed
  once, not a multi-period trend confirmed over time.
- **The playbook's reason-code thresholds are simple, un-tuned cutoffs** (percentile bands, a
  fixed 0.6 score threshold) — they are a starting shape for the queue, not thresholds validated
  against real editor decisions or downstream traffic outcomes yet.

In [ ]:
held_out_clients = len(unique_clients) - len(train_clients)
held_out_rows = int(test_mask.sum())
full_catalog_items = 519_606  # dim_content row count, from the warehouse release guide (docs/ml-intern-dataset-and-lane-guide.md)
starter_sample_rows = len(df)

print("=== Limitations — supporting numbers ===")
print(f"Held-out clients used for the decline-rate finding: {held_out_clients} ({held_out_rows} pages)")
print(f"Starter sample size: {starter_sample_rows} rows")
print(f"Full warehouse catalog size (dim_content): {full_catalog_items} items")
print(f"Starter sample is {starter_sample_rows/full_catalog_items*100:.2f}% of the full catalog")

=== Limitations — supporting numbers ===
Held-out clients used for the decline-rate finding: 7 (7611 pages)
Starter sample size: 30000 rows
Full warehouse catalog size (dim_content): 519606 items
Starter sample is 5.77% of the full catalog


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Every page in the 30,000-row sample gets one of four actions, driven by real, observed signals
only (`position_tier`, `ctr_gap`, `is_declining_label`, `impressions_90d`, `cluster`) — no
invented fields:

<table style="width:100%; border-collapse: collapse; text-align: left; font-family: sans-serif; font-size: 13px; color: #000000; background-color: #ffffff; table-layout: fixed;">
  <thead>
    <tr style="border-bottom: 2px solid #333333; background-color: #ffffff;">
      <th style="padding: 8px; width: 8%;">Priority</th>
      <th style="padding: 8px; width: 12%;">Action</th>
      <th style="padding: 8px; width: 22%;">Reason code</th>
      <th style="padding: 8px; width: 30%;">Trigger</th>
      <th style="padding: 8px; width: 8%; text-align: right;">n</th>
      <th style="padding: 8px; width: 20%;">What a reviewer does</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #f4f4f4; border-bottom: 1px solid #e0e0e0;">
      <td style="padding: 8px;">1</td>
      <td style="padding: 8px;"><b>REFRESH</b></td>
      <td style="padding: 8px;"><code>DECAY_HIGH_VALUE</code></td>
      <td style="padding: 8px;">ranks <code>top_3</code> / <code>page_1</code>, declining &gt;20% MoM, CTR well below its tier's typical rate</td>
      <td style="padding: 8px; text-align: right;">1,447</td>
      <td style="padding: 8px;">update citations, rewrite outdated sections, retarget headings</td>
    </tr>
    <tr style="background-color: #ffffff; border-bottom: 1px solid #e0e0e0;">
      <td style="padding: 8px;">2</td>
      <td style="padding: 8px;"><b>BOOST</b></td>
      <td style="padding: 8px;"><code>STRIKING_DISTANCE_BOOST</code></td>
      <td style="padding: 8px;">position 11-20 ("striking distance"), CTR below tier expectation</td>
      <td style="padding: 8px; text-align: right;">4,348</td>
      <td style="padding: 8px;">cheap fix — rewrite meta title/description, add internal links</td>
    </tr>
    <tr style="background-color: #f4f4f4; border-bottom: 1px solid #e0e0e0;">
      <td style="padding: 8px;">3</td>
      <td style="padding: 8px;"><b>PRUNE</b></td>
      <td style="padding: 8px;"><code>PRUNE_OBSOLETE_THIN</code></td>
      <td style="padding: 8px;">deep/unranked, declining, bottom-quartile visibility</td>
      <td style="padding: 8px; text-align: right;">405</td>
      <td style="padding: 8px;">evaluate for 301 redirect or consolidation</td>
    </tr>
    <tr style="background-color: #ffffff; border-bottom: 1px solid #e0e0e0;">
      <td style="padding: 8px;">4</td>
      <td style="padding: 8px;"><b>MONITOR</b></td>
      <td style="padding: 8px;"><code>STABLE_NO_ACTION</code></td>
      <td style="padding: 8px;">no strong signal either way</td>
      <td style="padding: 8px; text-align: right;">23,800</td>
      <td style="padding: 8px;">no action this cycle</td>
    </tr>
  </tbody>
</table>

**23.4%** of the queue is flagged for **mandatory human sign-off** before any action — every
`PRUNE` candidate, every top-10%-visibility page, every score sitting in the uncertain band
around the REFRESH cutoff, and every page whose cluster fit is weak (negative silhouette).

**No-go list — never automated:** URL deletion or redirection without a human checking first;
core brand/legal pages (hard-excluded regardless of score); site navigation or URL-structure
changes based solely on this output.

Full reason-code logic, thresholds, and export code live in
[`work/notebooks/w07_action_playbook.ipynb`](../notebooks/w07_action_playbook.ipynb) in this repo.

In [ ]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
low_visibility_cut = df.loc[eligible, "impressions_90d"].quantile(0.25)

def assign_playbook_rules(row):
    if row["position_tier"] in ("top_3", "page_1") and row["is_declining_label"] == 1 and row["baseline_action_score"] >= 0.6:
        return "REFRESH", "DECAY_HIGH_VALUE", 1
    elif row["position_tier"] == "striking" and row["ctr_gap"] > 0:
        return "BOOST", "STRIKING_DISTANCE_BOOST", 2
    elif row["position_tier"] in ("deep", "no_data") and row["is_declining_label"] == 1 and row["impressions_90d"] <= low_visibility_cut:
        return "PRUNE", "PRUNE_OBSOLETE_THIN", 3
    else:
        return "MONITOR", "STABLE_NO_ACTION", 4

playbook = df.loc[X.index].copy()
playbook[["action", "reason_code", "priority"]] = playbook.apply(assign_playbook_rules, axis=1, result_type="expand")

impressions_p90 = playbook["impressions_90d"].quantile(0.90)
score_lo, score_hi = playbook["baseline_action_score"].quantile([0.55, 0.65])
playbook["requires_human_signoff"] = (
    (playbook["action"] == "PRUNE")
    | (playbook["impressions_90d"] >= impressions_p90)
    | (playbook["baseline_action_score"].between(score_lo, score_hi))
)

print("=== Section 6: Ranked Action Playbook (recomputed here for the paper) ===")
print(playbook["action"].value_counts())
print(f"\nFlagged for mandatory human sign-off: {playbook['requires_human_signoff'].mean()*100:.1f}%")

=== Section 6: Ranked Action Playbook (recomputed here for the paper) ===
action
MONITOR    23800
BOOST       4348
REFRESH     1447
PRUNE        405
Name: count, dtype: int64

Flagged for mandatory human sign-off: 21.4%


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Three figures, generated below from the real, re-run numbers in this notebook (not hand-typed):

1. **Validation drop** — leaky vs. honest silhouette score, showing why the client-grouped
   split matters.
2. **Baseline vs. model** — rule-buckets silhouette vs. KMeans silhouette, same feature space.
3. **Decline rate by cluster** — the 7-cluster range on held-out clients, smallest to largest.

Plus the Week-7 action-mix chart (`work/figures/action_distribution.png`), reused as-is from the
playbook notebook. All are saved to `work/figures/` for the deployed paper to embed directly.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

os.makedirs("work/figures", exist_ok=True)

INK = "#16232E"
TEAL = "#0E7C86"
AMBER = "#D9A441"
GRID = "#D8DEE3"
mpl.rcParams.update({
    "font.family": "DejaVu Sans", "text.color": INK, "axes.edgecolor": GRID,
    "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
})

# Fig 1 — validation drop
fig, ax = plt.subplots(figsize=(6, 4))
labels1 = ["Leaky\n(all-data fit+score)", "Honest\n(held-out clients)"]
vals1 = [sil_full_leaky, sil_honest]
bars = ax.bar(labels1, vals1, color=[AMBER, TEAL], edgecolor=INK, linewidth=0.8, width=0.55)
for b, v in zip(bars, vals1):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.3f}", ha="center", fontsize=11, fontweight="bold")
ax.set_ylabel("Silhouette score")
ax.set_title("Validation drop: leaky vs. honest client split", fontweight="bold")
ax.set_ylim(0, max(vals1) * 1.25)
ax.grid(axis="y", color=GRID, linestyle="--", linewidth=0.7)
ax.set_axisbelow(True)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig("work/figures/validation_drop.png", dpi=300)
plt.close()

# Fig 2 — baseline vs model
fig, ax = plt.subplots(figsize=(6, 4))
labels2 = ["Baseline\n(rule buckets)", "KMeans\n(honest split)"]
vals2 = [baseline_sil, sil_honest]
bars = ax.bar(labels2, vals2, color=["#9AA5AD", TEAL], edgecolor=INK, linewidth=0.8, width=0.55)
for b, v in zip(bars, vals2):
    ax.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.4f}", ha="center", fontsize=11, fontweight="bold")
ax.set_ylabel("Silhouette score")
ax.set_title("Model vs. baseline, same feature space", fontweight="bold")
ax.set_ylim(0, max(vals2) * 1.3)
ax.grid(axis="y", color=GRID, linestyle="--", linewidth=0.7)
ax.set_axisbelow(True)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig("work/figures/baseline_vs_model.png", dpi=300)
plt.close()

# Fig 3 — decline rate by cluster (held-out clients)
fig, ax = plt.subplots(figsize=(7, 4))
dr = decline_rate.reset_index()
colors3 = [TEAL if v < 35 else (AMBER if v < 55 else "#C2453D") for v in dr["decline_pct"]]
bars = ax.barh(dr["cluster"].astype(str), dr["decline_pct"], color=colors3, edgecolor=INK, linewidth=0.8)
for b, v, n in zip(bars, dr["decline_pct"], dr["count"]):
    ax.text(v + 1.5, b.get_y() + b.get_height()/2, f"{v}% (n={n})", va="center", fontsize=9)
ax.set_xlabel("Decline rate (%)")
ax.set_ylabel("Cluster")
ax.set_title("Decline rate by cluster — held-out clients only", fontweight="bold")
ax.set_xlim(0, 75)
ax.grid(axis="x", color=GRID, linestyle="--", linewidth=0.7)
ax.set_axisbelow(True)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig("work/figures/decline_by_cluster.png", dpi=300)
plt.close()

print("Saved: work/figures/validation_drop.png, baseline_vs_model.png, decline_by_cluster.png")
print("(work/figures/action_distribution.png already exists from w07_action_playbook.ipynb)")

Saved: work/figures/validation_drop.png, baseline_vs_model.png, decline_by_cluster.png
(work/figures/action_distribution.png already exists from w07_action_playbook.ipynb)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 5-Minute Demo Outline

### 0:00–0:45 — Question & FlyRank Content Problem
- Start with the real decision: when a content team has hundreds of thousands of pages but limited review time, which pages should a strategist look at first?
- Explain that the goal was to group pages by observed behavior so the review queue is sharper than a single hand-written rule.
- Emphasize that this is decision-support for human review, not an automated content-action system.

### 0:45–1:45 — Method
- Main sample: 30,000 pseudonymized content items from the FlyRank ML Internship warehouse release.
- Five observed trailing-90-day signals: impressions, CTR, average ranking position, engagement rate, and content age.
- Standardize the features and use KMeans clustering.
- Compare it with a transparent CTR-gap/visibility rule-based baseline.
- Choose k=7 using silhouette search and validate honestly by holding out entire clients rather than random rows.

### 1:45–2:45 — One Chart
- Show the **baseline vs. model silhouette chart**: `work/figures/baseline_vs_model.png`.
- Point out that the rule-based baseline scored **0.0013**, while honest client-held-out KMeans scored **0.3046**.
- Explain that the comparison is on the same feature space and that silhouette measures separation, not prediction accuracy.

### 2:45–3:45 — One Honest Result
- The honest KMeans result remained substantially stronger than the rule-based baseline, suggesting that the five observed signals contain useful behavioral structure for grouping pages.
- The validation score was lower than the leaky all-data score of 0.3763, which is expected when the model is tested on clients it never saw.
- The decline-rate differences between clusters are descriptive associations only; they do not prove that a cluster causes pages to decline.

### 3:45–4:30 — Recommendation
- Turn the observed patterns into a human-reviewed priority queue:
  1. REFRESH high-value pages showing decline and weak CTR.
  2. BOOST pages in striking distance with below-expected CTR.
  3. PRUNE only as a review candidate for deep, declining, low-visibility pages.
  4. MONITOR pages without a strong signal.
- Require human sign-off before any consequential action, especially pruning or redirecting pages.

### 4:30–5:00 — Close
- State the main takeaway: behavioral clustering can provide a more structured starting point for content review than a single rule, but it has not been proven to improve traffic outcomes.
- Mention the key limitations: 30,000-row starter sample, 7 held-out clients for the decline-rate analysis, one snapshot, and no causal experiment.
- End with the decision-support framing: the model helps a human decide where to look first; it does not decide what happens to a page.

## Shareable Cuts

### Social Post

I built an unsupervised content-archetype clustering pipeline during my FlyRank ML Internship to explore a practical question: when a content team has too many pages to review manually, can observed behavior help prioritize where a human should look first?

Using 30,000 pseudonymized content items and five trailing-90-day signals — impressions, CTR, ranking position, engagement, and content age — I compared KMeans clustering with a transparent rule-based baseline. The honest client-held-out KMeans silhouette score was **0.3046**, compared with **0.0013** for the baseline, suggesting substantially stronger behavioral separation.

The important part is the framing: this is decision-support, not a causal model or an automated SEO system. The recommendations are designed to help a reviewer prioritize pages for refresh, boost, prune-review, or monitoring, with human sign-off before consequential actions.

### Employer-Facing Summary

I built an unsupervised KMeans content-archetype clustering pipeline to help prioritize which pages a content strategist should review first, and compared it against a transparent rule-based baseline. The analysis used 30,000 pseudonymized content items from the FlyRank ML Internship warehouse release, using impressions, CTR, ranking position, engagement rate, and content age as observed behavioral features. On an honest client-held-out validation, KMeans achieved a silhouette score of **0.3046** versus **0.0013** for the baseline, showing stronger behavioral separation while remaining a decision-support analysis rather than a causal or automated production system.